# Notebook discipline

**Course:** C02, Python for Practical AI Engineering · **Lesson:** [C02-M06-L01 Notebook discipline](https://learning.nextia-ai.com/courses/python/m06/notebook-discipline/)

**You will learn to** find the hidden state that makes a notebook give a different result each time, and make a notebook that gives the same result from a fresh start.

**Time:** about 30 minutes. **Needs:** no account and no packages beyond Python itself.

**Tested:** 2026-10-06, Python 3.12 and 3.14 in JupyterLab 4.6, from a fresh start (*Restart and run all*). Data: the course's synthetic tickets ([dataset card](https://github.com/usmanahmed99/nextia-learning-labs/blob/main/C02/data/dataset.md)).

## Setup

Run the next cell. It downloads the course's ticket file (1 KB) and checks that it is the expected version.

You should see `15 rows, checksum OK`. If you see a network error, check your internet connection, and run the cell again.

In [ ]:
import csv
import hashlib
import urllib.request
from pathlib import Path

DATA_URL = "https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/C02/data/tickets.csv"
DATA_SHA256 = "82e3fb4fa6d94a47b9a7b05d940a088596363e4dd11e89bf82ed486bd2325a1a"
DATA_FILE = Path("tickets.csv")

if not DATA_FILE.exists():
    urllib.request.urlretrieve(DATA_URL, DATA_FILE)
checksum = hashlib.sha256(DATA_FILE.read_bytes()).hexdigest()
assert checksum == DATA_SHA256, "tickets.csv is not the expected file. Delete it, then run this cell again."

with open(DATA_FILE, encoding="utf-8-sig", newline="") as file:
    rows = list(csv.DictReader(file))
print(len(rows), "rows, checksum OK")

## The idea: the kernel remembers, the page does not

A notebook has two parts. The **page** shows your cells. The **kernel** is the Python program that runs them, and it keeps every variable in memory until it restarts.

The kernel does not care about the order of the cells on the page. It runs a cell when you run it, with the variables that exist **at that moment**. The number in `[ ]` next to a cell shows when it ran: `[5]` ran after `[4]`, wherever it is on the page.

So the page can show one story while the kernel holds a different one. The next cells show how.

Run the next cell. It chooses which status to count.

In [ ]:
wanted_status = "open"

Run the next cell. It counts the rows with the wanted status, after cleaning the text.

You should see `open 9`.

In [ ]:
selected = [row for row in rows if row["status"].strip().lower() == wanted_status]
print(wanted_status, len(selected))

## Predict, then run

1. Go back to the cell `wanted_status = "open"`. Change it to `wanted_status = "closed"`, and run **only that cell**.
2. Do not run the count cell again.
3. **Predict:** what does the next cell print? Write your prediction down first.
4. Run the next cell.

In [ ]:
print(wanted_status, len(selected))

It prints `closed 9`. But only 3 tickets are closed. `wanted_status` is new, and `selected` is still the result for `"open"`, because the cell that makes `selected` did not run again. The page shows `"closed"` everywhere, so a reader believes that 9 tickets are closed.

This is **hidden state**: values in the kernel that the page does not explain. Change `wanted_status` back to `"open"` before you continue.

## A deleted cell that still counts

Run the next cell. Then **delete it**: select the cell, and use the delete command of your notebook (for example **Edit › Delete Cells** in JupyterLab, or the bin icon in Colab).

In [ ]:
priority_limit = 2

Run the next cell. It still works, although the cell that made `priority_limit` is gone: the kernel remembers it.

In [ ]:
urgent_enough = [row for row in rows if row["priority"].strip() in {"1", str(priority_limit)}]
print(len(urgent_enough), "rows with priority 1 or", priority_limit)

## Failure case: Restart and run all

Now give the notebook a fresh start, as a colleague who opens it would have:

- **JupyterLab:** Kernel › Restart Kernel and Run All Cells.
- **Colab:** Runtime › Restart session and run all.
- **Kaggle:** Run › Restart & clear cell outputs, then Run › Run all.

If you deleted the `priority_limit = 2` cell, the run stops at the cell above with:

```
NameError: name 'priority_limit' is not defined
```

The notebook worked only because of something that is no longer on the page. **Fix:** put the value back in a cell **above** the cell that uses it, then restart and run all again, until it runs from top to bottom without an error. Do this before you share a notebook, every time.

## Move reusable code out of the notebook

A function that you want to keep, test and use in a program belongs in a module, not in a cell. The cell magic `%%writefile` saves a cell as a file. Run the next cell: it makes the module `ticket_rules.py` next to the notebook.

In [ ]:
%%writefile ticket_rules.py
RULES_VERSION = 1


def clean_status(value: str) -> str:
    """Return the status with no outer spaces, in small letters."""
    return value.strip().lower()

Run the next cell. It imports the module and uses it. You should see `1 open`.

In [ ]:
import ticket_rules

print(ticket_rules.RULES_VERSION, ticket_rules.clean_status("  Open "))

## Change one thing: a new version of the module

Run the next cell. It saves version 2 of the module, which adds a function. Then run the cell after it, which imports the module again.

**Predict:** which version number does it print?

In [ ]:
%%writefile ticket_rules.py
RULES_VERSION = 2


def clean_status(value: str) -> str:
    """Return the status with no outer spaces, in small letters."""
    return value.strip().lower()


def is_closed(value: str) -> bool:
    """Return True if the cleaned status is closed."""
    return clean_status(value) == "closed"

In [ ]:
import ticket_rules

print(ticket_rules.RULES_VERSION, hasattr(ticket_rules, "is_closed"))

It prints `1 False`: version 1, without the new function. Python imports a module only once per kernel. The second `import` uses the copy in memory, so the kernel still has version 1, although the file has version 2. This is hidden state too.

Two fixes: restart the kernel (then every import reads the files again), or reload the one module, as in the next cell. You should see `2 True`.

In [ ]:
import importlib

importlib.reload(ticket_rules)
print(ticket_rules.RULES_VERSION, hasattr(ticket_rules, "is_closed"))

## Check yourself

Each exercise ends with a cell of `assert` lines. If your answer is correct, the cell prints `Correct.` If not, it shows a message that says what to check.

**Exercise 1.** In the next cell, count the rows with the status `pending` (after cleaning) into `pending_count`.

In [ ]:
wanted_status = "pending"
pending_count = ...  # Your code here: count the rows with the wanted status.

In [ ]:
assert pending_count == 2, f"pending_count is {pending_count!r}. Count the rows whose cleaned status is 'pending'. There are 2."
print("Correct.")

**Exercise 2.** Add a function `is_known_status(value)` to the module. It returns `True` if the cleaned value is `open`, `pending` or `closed`. Save the module with the next cell. The cell after it reloads the module and checks your function: 14 of the 15 rows have a known status (row 7 has `opne`).

In [ ]:
%%writefile ticket_rules.py
RULES_VERSION = 3
STATUSES = {"open", "pending", "closed"}


def clean_status(value: str) -> str:
    """Return the status with no outer spaces, in small letters."""
    return value.strip().lower()


# Your code here: add the function is_known_status(value).

In [ ]:
importlib.reload(ticket_rules)

known = [row for row in rows if ticket_rules.is_known_status(row["status"])]
assert ticket_rules.RULES_VERSION == 3, "Run the %%writefile cell above first, then this cell."
assert len(known) == 14, f"{len(known)} rows have a known status. There must be 14."
assert not ticket_rules.is_known_status("opne"), "'opne' is not a known status."
print("Correct.")

## Recap

- The kernel keeps every variable until it restarts. The order in which you ran the cells is what counts, not their order on the page.
- A changed cell that you did not run again, and a deleted cell, both leave **hidden state**.
- Before you share a notebook, **restart and run all**. It must run from top to bottom without an error.
- Keep exploration in the notebook. Move code that you want to keep into a module. After you change a module, reload it or restart the kernel.

Next lesson: [Command-line interface](https://learning.nextia-ai.com/courses/python/m06/command-line-interface/).